## Travel Planning Multi-Agent System
Supervisor agent coordinates Flight Agent and Hotel Agent.

In [11]:
#Configure AWS Bedrock credentials
import os
# os.environ["AWS_ACCESS_KEY_ID"]="AKIAJGXGUDKJ"
# os.environ["AWS_SECRET_ACCESS_KEY"]= "wQMJSFKPssSaJTsKtCM2LR/RgCmQ6LRd1h" "amazon.nova-lite-v1:0"
# ---------------------------
# AWS Bedrock LLM Setup
# ---------------------------
model = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", 
                            region_name="us-east-1",
                            temperature=0.5, 
                            max_tokens=500)

In [ ]:
"""
Travel Planning Multi-Agent System
Supervisor agent coordinates Flight Agent and Hotel Agent.
"""

from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# ============================================================================
# Define low-level API tools
# ============================================================================

@tool
def search_flights(origin: str, destination: str, date: str) -> list[dict]:
   """Return stubbed list of flights."""
   return [
       {"flight": "SG101", "time": "08:00", "price": 350},
       {"flight": "SG202", "time": "11:00", "price": 300},
       {"flight": "SG303", "time": "18:00", "price": 420},
   ]


@tool
def book_flight(flight_id: str, passenger: str) -> str:
   """Book a flight."""
   return f"Flight {flight_id} booked for {passenger}"


@tool
def search_hotels(city: str, checkin: str, nights: int) -> list[dict]:
   """Return stubbed list of hotels."""
   return [
       {"name": "Marina Bay Hotel", "price": 180},
       {"name": "City Center Inn", "price": 120},
       {"name": "Harbor View Suites", "price": 150},
   ]


@tool
def book_hotel(hotel_name: str, nights: int, guest: str) -> str:
   """Book a hotel."""
   return f"Hotel '{hotel_name}' booked for {guest} for {nights} nights"


# ============================================================================
# Create specialized sub-agents
# ============================================================================


# Flight Agent ---------------------------------------------------------------

flight_agent = create_react_agent(
   model,
   tools=[search_flights, book_flight],
   prompt=(
       "You are a flight booking assistant. "
       "Parse travel requests (origin, destination, dates). "
       "Use search_flights to get flight options. "
       "Choose a good option and use book_flight. "
       "Always confirm the final booked flight."
   )
)

# Hotel Agent ---------------------------------------------------------------

hotel_agent = create_react_agent(
   model,
   tools=[search_hotels, book_hotel],
   prompt=(
       "You are a hotel booking assistant. "
       "Parse lodging requests (city, dates, nights). "
       "Use search_hotels to get options. "
       "Choose a suitable hotel and use book_hotel. "
       "Always confirm what was booked."
   )
)

# ============================================================================
# Wrap sub-agents as tools for the supervisor
# ============================================================================

@tool
def arrange_flight(request: str) -> str:
   """Use natural language to book a flight."""
   result = flight_agent.invoke({
       "messages": [{"role": "user", "content": request}]
   })
   return result["messages"][-1].text


@tool
def arrange_hotel(request: str) -> str:
   """Use natural language to book a hotel."""
   result = hotel_agent.invoke({
       "messages": [{"role": "user", "content": request}]
   })
   return result["messages"][-1].text


# ============================================================================
# Create the supervisor agent
# ============================================================================

supervisor_agent = create_react_agent(
   model,
   tools=[arrange_flight, arrange_hotel],
   prompt=(
       "You are a travel planning assistant. "
       "You can book flights and hotels. "
       "Break down requests and call the appropriate tools. "
       "For multi-step tasks, call multiple tools sequentially."
   )
)

In [12]:
# Example 1
if __name__ == "__main__":
   user_request = (
       "Plan a 3-day trip to Singapore next month. "
       "Book a morning flight from Mumbai and reserve a hotel near Marina Bay."
   )

   print("\nUser Request:\n", user_request)
   print("\n" + "="*80 + "\n")

   for step in supervisor_agent.stream(
       {"messages": [{"role": "user", "content": user_request}]}
   ):
       for update in step.values():
           for message in update.get("messages", []):
               message.pretty_print()



User Request:
 Plan a 3-day trip to Singapore next month. Book a morning flight from Mumbai and reserve a hotel near Marina Bay.


================================== Ai Message ==================================

[{'type': 'text', 'text': 'I will first book a morning flight from Mumbai to Singapore. Then I will book a hotel near Marina Bay for three days.'}, {'type': 'tool_use', 'name': 'arrange_flight', 'input': {'request': 'Book a morning flight from Mumbai to Singapore for one person next month.'}, 'id': 'tooluse_XLX1paMxS9G3InybiyZkJg'}]
Tool Calls:
  arrange_flight (tooluse_XLX1paMxS9G3InybiyZkJg)
 Call ID: tooluse_XLX1paMxS9G3InybiyZkJg
  Args:
    request: Book a morning flight from Mumbai to Singapore for one person next month.
================================= Tool Message =================================
Name: arrange_flight

<bound method BaseMessage.text of AIMessage(content='I have booked you on flight SG101 from Mumbai to Singapore, leaving at 8am next month.', addition

In [13]:
# Example 2
# ============================================================================
from rich import print

if __name__ == "__main__":
   user_request = (
       "Plan a 3-day trip to Malysia next month. "
       "Book a morning flight from Bangalore and reserve a hotel near City center."
   )

   print("\nUser Request:\n", user_request)
   print("\n" + "="*80 + "\n")

   for step in supervisor_agent.stream(
       {"messages": [{"role": "user", "content": user_request}]}
   ):
       for update in step.values():
           for message in update.get("messages", []):
               message.pretty_print()

User Request:
 Plan a 3-day trip to Malysia next month. Book a morning flight from Bangalore and reserve a hotel near City 
center.

================================================================================

================================== Ai Message ==================================

[{'type': 'text', 'text': 'I will first find a morning flight from Bangalore to Malaysia. Then, I will book a hotel in Malaysia near the city centre.'}, {'type': 'tool_use', 'name': 'arrange_flight', 'input': {'request': 'Book a morning flight from Bangalore to Malaysia next month.'}, 'id': 'tooluse_LsX9F3WfQJaVI0EsIy4eBw'}]
Tool Calls:
  arrange_flight (tooluse_LsX9F3WfQJaVI0EsIy4eBw)
 Call ID: tooluse_LsX9F3WfQJaVI0EsIy4eBw
  Args:
    request: Book a morning flight from Bangalore to Malaysia next month.
================================= Tool Message =================================
Name: arrange_flight

<bound method BaseMessage.text of AIMessage(content='Flight SG202 has been booked for your journey from Bangalore to Malaysia next month. The flight is at 11:00 and cost 300.', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '54307dc3-d374-42e6-962b-fc04e6fb620d', 'HTTPStatus